Python Notebook for training and testing purposes

In [ ]:
from fastai.vision.all import *
import cv2
import numpy as np
from pathlib import Path

In [ ]:
path = Path('datasets/open_setup/optical_images1')

dls = DataBlock(
    blocks=(ImageBlock, CategoryBlock), 
    get_items=get_image_files, 
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=parent_label,
).dataloaders(path, bs=32)




In [ ]:
dls.show_batch(max_n=6)


In [ ]:
learn = vision_learner(dls, densenet121, metrics=accuracy)


In [ ]:
learn.fine_tune(5)


In [ ]:
#learn.export("my_model.pkl")

In [ ]:
from fastai.vision.all import *
from fastai.callback.tracker import SaveModelCallback
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import json
from pathlib import Path
import os
 
# Define model architectures and preprocessing options
models = [resnet34, resnet18, squeezenet1_0, squeezenet1_1, densenet121]
model_names = ['resnet34', 'resnet18', 'squeezenet1_0', 'squeezenet1_1', 'densenet121']
folders = ['optical_images1', 'optical_images2']
crop_options = [False, True]
 
# Number of folds for cross-validation
n_folds = 5
epochs = 5
 
results = []
os.makedirs('experiments', exist_ok=True)
 
for folder in folders:
    for crop in crop_options:
        path = Path(f'datasets/open_setup/{folder}')
        files = get_image_files(path)
        labels = [parent_label(f) for f in files]
        classes = sorted(list(set(labels)))
        class2idx = {c: i for i, c in enumerate(classes)}
        y_numeric = np.array([class2idx[l] for l in labels])
        
        for model_func, model_name in zip(models, model_names):
            experiment_name = f"experiments/{folder}_{'crop' if crop else 'nocrop'}_{model_name}"
            os.makedirs(experiment_name, exist_ok=True)
            
            all_fold_acc = []
            all_fold_cm = []
            kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
            
            for fold, (train_idx, valid_idx) in enumerate(kf.split(files), 1):
                print(f"{experiment_name} | Fold {fold}")
                train_files = [files[i] for i in train_idx]
                valid_files = [files[i] for i in valid_idx]
                
                # Preprocessing
                if crop:
                    item_tfms = [Resize(192, method='squish')]
                else:
                    item_tfms = None
                
                dblock = DataBlock(
                    blocks=(ImageBlock, CategoryBlock),
                    get_items=lambda _: train_files + valid_files,
                    splitter=IndexSplitter(valid_idx),
                    get_y=parent_label,
                    item_tfms=item_tfms if item_tfms else []
)
                dls = dblock.dataloaders(path, bs=32)
                
                learn = vision_learner(dls, model_func, metrics=[accuracy], pretrained=True)
                
                learn.fine_tune(epochs, cbs=SaveModelCallback(monitor="valid_loss", fname=f"{experiment_name}/best_model_fold{fold}"))
                learn.load(f"{experiment_name}/best_model_fold{fold}")
                
                val_loss, acc = learn.validate()[:2]
                all_fold_acc.append(float(acc))
                
                # Confusion matrix
                preds, targs = learn.get_preds(dl=dls.valid)
                pred_labels = preds.argmax(dim=1).numpy()
                targ_labels = targs.numpy()
                cm = confusion_matrix(targ_labels, pred_labels)
                all_fold_cm.append(cm)
                
                # Plot and save confusion matrix
                disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
                fig, ax = plt.subplots(figsize=(5, 5))
                disp.plot(ax=ax, cmap=plt.cm.Blues, xticks_rotation='vertical')
                plt.title(f"{experiment_name} - Fold {fold}")
                plt.tight_layout()
                plt.savefig(f"{experiment_name}/confusion_matrix_fold{fold}.png")
                plt.close()
            
            # Aggregate results
            avg_acc = np.mean(all_fold_acc)
            results.append({
                'experiment': experiment_name,
                'folder': folder,
                'crop': crop,
                'model': model_name,
                'accuracy': avg_acc,
                'fold_accuracies': all_fold_acc,
            })
            
            # Save results for this experiment
            with open(f"{experiment_name}/results.json", "w") as f:
                json.dump(results[-1], f, indent=4)
 
# Save all results to CSV
results_df = pd.DataFrame(results)
results_df.to_csv('model_preprocessing_results_cv.csv', index=False)
print('All results saved to model_preprocessing_results_cv.csv')
 
# Print comparison table
print("\n── Results ──────────────────────────────────────────")
print(f"{'Experiment':<50} {'Accuracy':>10}")
for r in sorted(results, key=lambda x: x['accuracy'], reverse=True):
    print(f"{r['experiment']:<50} {r['accuracy']:>10.4f}")

In [ ]:
from fastai.vision.all import *
from fastai.callback.tracker import SaveModelCallback
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import json
import multiprocessing
from pathlib import Path
import os

models      = [resnet34, resnet18, squeezenet1_0, squeezenet1_1, densenet121]
model_names = ['resnet34', 'resnet18', 'squeezenet1_0', 'squeezenet1_1', 'densenet121']
folders     = ['optical_images1', 'optical_images2']
crop_options = [False, True]

n_folds     = 5
epochs      = 5
NUM_WORKERS = min(multiprocessing.cpu_count(), 8)
BATCH_SIZE  = 64

results = []
os.makedirs('experiments', exist_ok=True)

for folder in folders:
    for crop in crop_options:
        path   = Path(f'datasets/open_setup/{folder}')
        files  = get_image_files(path)
        labels = [parent_label(f) for f in files]
        classes = sorted(set(labels))

        for model_func, model_name in zip(models, model_names):
            experiment_name = f"experiments/{folder}_{'crop' if crop else 'nocrop'}_{model_name}"
            os.makedirs(experiment_name, exist_ok=True)

            all_fold_acc = []
            all_fold_cm  = []
            kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

            for fold, (train_idx, valid_idx) in enumerate(kf.split(files), 1):
                print(f"\n{experiment_name} | Fold {fold}/{n_folds}")

                train_files = [files[i] for i in train_idx]
                valid_files = [files[i] for i in valid_idx]

                # Fix: indices relative to the combined list
                n_train         = len(train_files)
                valid_idx_local = list(range(n_train, n_train + len(valid_files)))

                item_tfms = [Resize(192, method='squish')] if crop else []

                dblock = DataBlock(
                    blocks=(ImageBlock, CategoryBlock),
                    get_items=lambda _: train_files + valid_files,
                    splitter=IndexSplitter(valid_idx_local),
                    get_y=parent_label,
                    item_tfms=item_tfms
                )
                dls = dblock.dataloaders(
                    path,
                    bs=BATCH_SIZE,
                    num_workers=NUM_WORKERS   # parallelise data loading
                )

                learn = vision_learner(dls, model_func, metrics=[accuracy], pretrained=True)
                learn = learn.to_fp16()      # mixed precision
                learn.path = Path(experiment_name)  # fix SaveModelCallback path

                learn.fine_tune(
                    epochs,
                    cbs=SaveModelCallback(monitor="valid_loss",
                                         fname=f"best_model_fold{fold}")
                )
                learn.load(f"best_model_fold{fold}")

                val_loss, acc = learn.validate()[:2]
                all_fold_acc.append(float(acc))

                preds, targs = learn.get_preds(dl=dls.valid)
                pred_labels  = preds.argmax(dim=1).numpy()
                targ_labels  = targs.numpy()
                cm = confusion_matrix(targ_labels, pred_labels)
                all_fold_cm.append(cm)

                disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                              display_labels=classes)
                fig, ax = plt.subplots(figsize=(5, 5))
                disp.plot(ax=ax, cmap=plt.cm.Blues, xticks_rotation='vertical')
                plt.title(f"{experiment_name} - Fold {fold}")
                plt.tight_layout()
                plt.savefig(f"{experiment_name}/confusion_matrix_fold{fold}.png")
                plt.close()

            avg_acc = np.mean(all_fold_acc)
            results.append({
                'experiment':      experiment_name,
                'folder':          folder,
                'crop':            crop,
                'model':           model_name,
                'accuracy':        avg_acc,
                'fold_accuracies': all_fold_acc,
            })
            with open(f"{experiment_name}/results.json", "w") as f:
                json.dump(results[-1], f, indent=4)

            print(f"\n[DONE] {experiment_name}  avg_acc={avg_acc:.4f}")

results_df = pd.DataFrame(results)
results_df.to_csv('model_preprocessing_results_cv.csv', index=False)

print("\n── Results ──────────────────────────────────────────")
print(f"{'Experiment':<55} {'Avg Accuracy':>12}")
for r in sorted(results, key=lambda x: x['accuracy'], reverse=True):
    print(f"{r['experiment']:<55} {r['accuracy']:>12.4f}")

In [3]:
import os
import json
import glob

# Path to experiment folders
base_dir = 'experiments'

# Find all results.json files (one per experiment)
json_files = glob.glob(os.path.join(base_dir, '**', 'results.json'), recursive=True)

fold_accuracies = []

for file in json_files:
    with open(file, 'r') as f:
        data = json.load(f)
        experiment = data.get('experiment', file)
        # Each experiment's fold accuracies
        for i, acc in enumerate(data.get('fold_accuracies', []), 1):
            fold_accuracies.append({
                'experiment': experiment,
                'fold': i,
                'accuracy': acc
            })

# Sort by accuracy descending
fold_accuracies.sort(key=lambda x: x['accuracy'], reverse=True)

# Print results
print(f"{'Experiment':<60} {'Fold':>4} {'Accuracy':>10}")
for entry in fold_accuracies:
    print(f"{entry['experiment']:<60} {entry['fold']:>4} {entry['accuracy']:>10.4f}")


Experiment                                                   Fold   Accuracy
experiments/optical_images2_crop_densenet121                    1     1.0000
experiments/optical_images2_crop_resnet18                       5     1.0000
experiments/optical_images2_crop_resnet34                       4     1.0000
experiments/optical_images2_crop_squeezenet1_0                  4     1.0000
experiments/optical_images2_crop_squeezenet1_1                  1     1.0000
experiments/optical_images2_crop_squeezenet1_1                  4     1.0000
experiments/optical_images2_crop_squeezenet1_1                  5     1.0000
experiments/optical_images2_nocrop_densenet121                  1     1.0000
experiments/optical_images2_nocrop_resnet18                     4     1.0000
experiments/optical_images2_nocrop_resnet18                     5     1.0000
experiments/optical_images2_nocrop_resnet34                     4     1.0000
experiments/optical_images2_nocrop_squeezenet1_0                3     1.0000

In [5]:
import os
import json
import glob
from collections import defaultdict

# Path to experiment folders
base_dir = 'experiments'

# Find all results.json files (one per experiment)
json_files = glob.glob(os.path.join(base_dir, '**', 'results.json'), recursive=True)

model_scores = defaultdict(list)

for file in json_files:
    with open(file, 'r') as f:
        data = json.load(f)
        model = data.get('model', 'unknown')
        avg_acc = data.get('accuracy', None)
        if avg_acc is not None:
            model_scores[model].append(avg_acc)

# Compute average accuracy for each model
model_avg = []
for model, scores in model_scores.items():
    model_avg.append({'model': model, 'mean_accuracy': sum(scores)/len(scores), 'n_experiments': len(scores)})

# Sort by mean accuracy descending
model_avg.sort(key=lambda x: x['mean_accuracy'], reverse=True)

# Print results
print(f"{'Model':<20} {'Mean Accuracy':>15} {'# Experiments':>15}")
for entry in model_avg:
    print(f"{entry['model']:<20} {entry['mean_accuracy']:>15.4f} {entry['n_experiments']:>15}")


Model                  Mean Accuracy   # Experiments
squeezenet1_1                 0.9783               4
squeezenet1_0                 0.9726               4
resnet18                      0.9709               4
densenet121                   0.9669               4
resnet34                      0.9629               4


In [6]:
import os
import json
import glob

# Path to experiment folders
base_dir = 'experiments'

# Find all results.json files (one per experiment)
json_files = glob.glob(os.path.join(base_dir, '**', 'results.json'), recursive=True)

experiment_scores = []

for file in json_files:
    with open(file, 'r') as f:
        data = json.load(f)
        experiment = data.get('experiment', file)
        avg_acc = data.get('accuracy', None)
        if avg_acc is not None:
            experiment_scores.append({'experiment': experiment, 'mean_accuracy': avg_acc})

# Sort by mean accuracy descending
experiment_scores.sort(key=lambda x: x['mean_accuracy'], reverse=True)

# Print results
print(f"{'Experiment':<60} {'Mean Accuracy':>15}")
for entry in experiment_scores:
    print(f"{entry['experiment']:<60} {entry['mean_accuracy']:>15.4f}")


Experiment                                                     Mean Accuracy
experiments/optical_images2_crop_squeezenet1_1                        0.9909
experiments/optical_images2_crop_squeezenet1_0                        0.9909
experiments/optical_images2_nocrop_squeezenet1_1                      0.9909
experiments/optical_images2_nocrop_resnet18                           0.9886
experiments/optical_images2_crop_resnet18                             0.9863
experiments/optical_images2_crop_resnet34                             0.9863
experiments/optical_images2_nocrop_resnet34                           0.9863
experiments/optical_images2_nocrop_squeezenet1_0                      0.9863
experiments/optical_images2_nocrop_densenet121                        0.9840
experiments/optical_images2_crop_densenet121                          0.9818
experiments/optical_images1_crop_squeezenet1_1                        0.9749
experiments/optical_images1_crop_squeezenet1_0                        0.9589